# Calibration AR Notebook - From Measurement to Image-Aligned Augmentation

**Applied Computer Vision, TU Berlin - 2026**

Most of Lab 2 treats the camera as a measuring instrument. We use known board geometry and image correspondences to infer something about the real world.

This extension runs that logic in the reverse direction. Once we trust the camera model and the board plane, we can place virtual 3D geometry back into the image in a perspectively consistent way. If the camera model is poor, that consistency breaks visibly.

This notebook expects two things to exist already:

- `../captured_points/<board>/intrinsics.yml`
- `../data/<board>/board_clip.mp4`

The default path here is **ChArUco first** because a short moving clip is easier to handle when the board does not have to stay perfectly front-facing in every frame. If needed, you can switch the notebook to checkerboard mode in the setup cell.


In [ ]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Video, display

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT_DIR = NOTEBOOK_DIR.parent
SRC_DIR = ROOT_DIR / 'src'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from board_config import (
    SQUARE_SIZE_M,
    board_clip_path,
    board_intrinsics_path,
    normalize_board_name,
)
from ar_helpers import (
    annotate_frame,
    detect_board,
    draw_axes,
    draw_cuboid,
    draw_detected_points,
    estimate_pose,
    get_video_info,
    load_intrinsics,
    make_centered_cuboid,
    read_frame,
    read_frames,
    render_overlay_video,
    sample_frame_indices,
    track_poses,
)

plt.style.use('default')
np.set_printoptions(suppress=True, precision=4)

BOARD_TYPE = normalize_board_name('charuco')  # change to 'checker' to use a checkerboard clip
VIDEO_PATH = board_clip_path(BOARD_TYPE)
YAML_PATH = board_intrinsics_path(BOARD_TYPE)
OUTPUT_DIR = NOTEBOOK_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_COUNT = 9
AXIS_LENGTH_M = 2.0 * SQUARE_SIZE_M
CUBOID_SIZE_X = 2.5 * SQUARE_SIZE_M
CUBOID_SIZE_Y = 2.0 * SQUARE_SIZE_M
CUBOID_HEIGHT_M = 2.0 * SQUARE_SIZE_M

if not VIDEO_PATH.exists():
    raise FileNotFoundError(f'Expected video at {VIDEO_PATH}')
if not YAML_PATH.exists():
    raise FileNotFoundError(f'Expected intrinsics at {YAML_PATH}')

K, d = load_intrinsics(YAML_PATH)
video_info = get_video_info(VIDEO_PATH)

print(f'Board type: {BOARD_TYPE}')
print(f'Video: {VIDEO_PATH}')
print(f'Intrinsics: {YAML_PATH}')
print(f"Frames: {video_info['frame_count']}")
print(f"FPS: {video_info['fps']:.2f}")
print(f"Resolution: {video_info['width']} x {video_info['height']}")
print(f"Duration: {video_info['duration_s']:.2f} s")
if video_info['frame_count_source'] == 'decoded':
    print('Frame count metadata was unreliable, so the notebook counted frames by decoding the clip once.')


def show_frame_grid(items, ncols=3, figsize=(14, 10)):
    if not items:
        raise ValueError('No frames are available to display. Check that the video contains readable frames.')

    n = len(items)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.atleast_1d(axes).ravel()

    for ax in axes[n:]:
        ax.axis('off')

    for ax, (title, frame_bgr) in zip(axes, items):
        ax.imshow(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis('off')

    plt.tight_layout()
    plt.show()


## 1. Load the clip and inspect one representative frame

The clip has already been captured on the Pi and copied to the laptop. We now work entirely from that local file.

The first step is just to look at one frame and make sure the board is visible in a way that should support detection and pose estimation.


In [ ]:
representative_index = video_info['frame_count'] // 2
representative_frame = read_frame(VIDEO_PATH, representative_index)

plt.figure(figsize=(7, 5))
plt.imshow(cv2.cvtColor(representative_frame, cv2.COLOR_BGR2RGB))
plt.title(f'Representative frame {representative_index}')
plt.axis('off')
plt.show()


## 2. Turn the clip into a small sampled storyboard

A short video is still just a sequence of images. Before we do any geometry, it is useful to sample a handful of frames so we can see how the board moves through the sequence.


In [ ]:
sample_indices = sample_frame_indices(video_info['frame_count'], num_samples=SAMPLE_COUNT)
sample_frames = read_frames(VIDEO_PATH, sample_indices)

show_frame_grid(
    [(f'Frame {idx}', frame) for idx, frame in sample_frames],
    ncols=3,
    figsize=(14, 10),
)


## 3. Detect the board in sampled frames

Before we can place anything virtual, we need stable 2D image evidence from the real scene.

In this extension the default ChArUco path is slightly more permissive than the main calibration-data collector. For pose estimation in a moving clip, we only need enough board correspondences to estimate pose reliably, not necessarily the full ChArUco corner set in every frame.


In [ ]:
sample_detections = []
valid_samples = 0

for idx, frame in sample_frames:
    detection = detect_board(frame, board_type=BOARD_TYPE)
    sample_detections.append((idx, frame, detection))
    if detection is not None:
        valid_samples += 1

print(f'Valid sampled detections: {valid_samples} / {len(sample_detections)}')

sample_detection_items = []
for idx, frame, detection in sample_detections:
    if detection is None:
        vis = annotate_frame(frame, 'Board not detected')
        title = f'Frame {idx} - no detection'
    else:
        vis = draw_detected_points(frame, detection)
        title = f"Frame {idx} - {len(detection['image_points'])} points"
    sample_detection_items.append((title, vis))

show_frame_grid(sample_detection_items, ncols=3, figsize=(14, 10))


## 4. Estimate pose and draw axes

The board gives us a coordinate frame in the world. Once we estimate pose, we can draw axes that are attached to that frame.

This is the conceptual bridge into augmentation. If the axes sit correctly on the board, then a more complex virtual object can also sit correctly on the board.


In [ ]:
poses, valid_pose_count = track_poses(VIDEO_PATH, BOARD_TYPE, K, d)
print(f'Valid poses across the full clip: {valid_pose_count} / {video_info["frame_count"]}')

sample_pose_items = []
for idx, frame in sample_frames:
    pose = poses[idx]
    if pose is None:
        vis = annotate_frame(frame, 'Pose unavailable')
        title = f'Frame {idx} - no pose'
    else:
        vis = draw_axes(frame, pose, K, d, axis_length_m=AXIS_LENGTH_M)
        title = f'Frame {idx} - axes'
    sample_pose_items.append((title, vis))

show_frame_grid(sample_pose_items, ncols=3, figsize=(14, 10))


## 5. Because we can draw axes, we can also draw a box

The axes tell us that we know the board pose for a frame. The next step is to define a simple cuboid in board coordinates and project it into the image.

This is the key conceptual reversal of the main lab. Earlier, the camera helped us infer geometry from the world. Here, the calibrated camera lets us place synthetic geometry back into the image in a perspectively consistent way.


In [ ]:
preview_index = None
for idx in sample_indices:
    if poses[idx] is not None:
        preview_index = idx
        break

if preview_index is None:
    raise RuntimeError('No valid pose was found in the sampled frames.')

preview_frame = read_frame(VIDEO_PATH, preview_index)
preview_pose = poses[preview_index]
vertices, edges = make_centered_cuboid(
    board_type=BOARD_TYPE,
    size_x=CUBOID_SIZE_X,
    size_y=CUBOID_SIZE_Y,
    height=CUBOID_HEIGHT_M,
)

preview_with_axes = draw_axes(preview_frame, preview_pose, K, d, axis_length_m=AXIS_LENGTH_M)
preview_with_cuboid = draw_cuboid(preview_with_axes, preview_pose, K, d, vertices, edges)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(cv2.cvtColor(preview_with_axes, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Axes on frame {preview_index}')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(preview_with_cuboid, cv2.COLOR_BGR2RGB))
axes[1].set_title('Axes plus wireframe cuboid')
axes[1].axis('off')

plt.tight_layout()
plt.show()


## 6. Render augmented videos and view them inline

Now we use the same pose estimates across the whole clip. First we save an axes preview video. Then we save a cuboid overlay video. Both stay on disk locally, and the notebook displays them inline without requiring you to jump to another tool.


In [ ]:
axes_video_path = OUTPUT_DIR / 'axes_preview.mp4'
cuboid_video_path = OUTPUT_DIR / 'cuboid_overlay.mp4'

render_overlay_video(
    VIDEO_PATH,
    axes_video_path,
    poses,
    K,
    d,
    axis_length_m=AXIS_LENGTH_M,
    draw_mode='axes',
)
render_overlay_video(
    VIDEO_PATH,
    cuboid_video_path,
    poses,
    K,
    d,
    vertices=vertices,
    edges=edges,
    axis_length_m=AXIS_LENGTH_M,
    draw_mode='cuboid',
)

print(f'Saved {axes_video_path}')
print(f'Saved {cuboid_video_path}')

display(Video(filename=str(axes_video_path), embed=False, width=640, html_attributes='controls loop'))
display(Video(filename=str(cuboid_video_path), embed=False, width=640, html_attributes='controls loop'))


## 7. Perturb the camera model deliberately

A good calibration should make the cuboid sit plausibly on the board. A poor camera model should damage that alignment.

In the next cell, adjust a few parameters. Nothing updates in real time. After you choose values, run the following cells to preview the effect on one frame and then rebuild the full augmented video.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

base_K = K.copy().astype(np.float64)
base_d = np.ravel(d).astype(np.float64)
if base_d.size < 5:
    base_d = np.pad(base_d, (0, 5 - base_d.size))

fx_scale_slider = widgets.FloatSlider(value=1.0, min=0.85, max=1.15, step=0.01, description='fx scale', continuous_update=False)
fy_scale_slider = widgets.FloatSlider(value=1.0, min=0.85, max=1.15, step=0.01, description='fy scale', continuous_update=False)
cx_shift_slider = widgets.FloatSlider(value=0.0, min=-40.0, max=40.0, step=1.0, description='cx shift', continuous_update=False)
cy_shift_slider = widgets.FloatSlider(value=0.0, min=-40.0, max=40.0, step=1.0, description='cy shift', continuous_update=False)
k1_delta_slider = widgets.FloatSlider(value=0.0, min=-0.20, max=0.20, step=0.01, description='k1 delta', continuous_update=False)
k2_delta_slider = widgets.FloatSlider(value=0.0, min=-0.20, max=0.20, step=0.01, description='k2 delta', continuous_update=False)
p1_delta_slider = widgets.FloatSlider(value=0.0, min=-0.03, max=0.03, step=0.001, description='p1 delta', continuous_update=False)
p2_delta_slider = widgets.FloatSlider(value=0.0, min=-0.03, max=0.03, step=0.001, description='p2 delta', continuous_update=False)
k3_delta_slider = widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.05, description='k3 delta', continuous_update=False)

sliders = widgets.VBox([
    fx_scale_slider,
    fy_scale_slider,
    cx_shift_slider,
    cy_shift_slider,
    k1_delta_slider,
    k2_delta_slider,
    p1_delta_slider,
    p2_delta_slider,
    k3_delta_slider,
])

display(sliders)


In [ ]:
trial_K = base_K.copy()
trial_K[0, 0] *= fx_scale_slider.value
trial_K[1, 1] *= fy_scale_slider.value
trial_K[0, 2] += cx_shift_slider.value
trial_K[1, 2] += cy_shift_slider.value

trial_d = base_d.copy()
trial_d[0] += k1_delta_slider.value
trial_d[1] += k2_delta_slider.value
trial_d[2] += p1_delta_slider.value
trial_d[3] += p2_delta_slider.value
trial_d[4] += k3_delta_slider.value

preview_original = draw_cuboid(preview_with_axes, preview_pose, K, d, vertices, edges)
preview_perturbed = draw_cuboid(
    draw_axes(preview_frame, preview_pose, trial_K, trial_d.reshape(-1, 1), axis_length_m=AXIS_LENGTH_M),
    preview_pose,
    trial_K,
    trial_d.reshape(-1, 1),
    vertices,
    edges,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(cv2.cvtColor(preview_original, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original camera model')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(preview_perturbed, cv2.COLOR_BGR2RGB))
axes[1].set_title('Perturbed camera model')
axes[1].axis('off')

plt.tight_layout()
plt.show()


## 8. Rebuild the whole cuboid video with the perturbed parameters

Run the next cell after you are satisfied with the parameter values. The pose track stays fixed, so the new video isolates what those camera-model changes do to the image-aligned augmentation.


In [ ]:
trial_K = base_K.copy()
trial_K[0, 0] *= fx_scale_slider.value
trial_K[1, 1] *= fy_scale_slider.value
trial_K[0, 2] += cx_shift_slider.value
trial_K[1, 2] += cy_shift_slider.value

trial_d = base_d.copy()
trial_d[0] += k1_delta_slider.value
trial_d[1] += k2_delta_slider.value
trial_d[2] += p1_delta_slider.value
trial_d[3] += p2_delta_slider.value
trial_d[4] += k3_delta_slider.value
trial_d = trial_d.reshape(-1, 1)

perturbed_video_path = OUTPUT_DIR / 'cuboid_overlay_perturbed.mp4'
render_overlay_video(
    VIDEO_PATH,
    perturbed_video_path,
    poses,
    trial_K,
    trial_d,
    vertices=vertices,
    edges=edges,
    axis_length_m=AXIS_LENGTH_M,
    draw_mode='cuboid',
)

print(f'Saved {perturbed_video_path}')
display(Video(filename=str(perturbed_video_path), embed=False, width=640, html_attributes='controls loop'))


## Reflection

Write a few short notes for yourself or for your lab notes.

1. Why does the cuboid stay attached to the board when the calibration and pose are good?
2. Which parameter changes damage the overlay most clearly, and where in the image do you notice that first?
3. Why is this a useful complement to the measurement story from the main lab, rather than a separate unrelated trick?
